# Faithfulness and Groundedness as Measurable Quantities — narrative notebook

This notebook walks the topic section by section, importing the canonical reference `faithfulness_groundedness.py` (which owns every number). HyDE modeled the quality of a generated answer with one hallucination rate $p$; here we *measure* it on the generated text as two numbers — **faithfulness** (the precision of an answer's atomic claims against the retrieved context) and **groundedness / coverage** (the recall of the supportable facts). Run top to bottom; the final cell re-runs the full assertion harness.

In [ ]:
import numpy as np
import faithfulness_groundedness as fg

corpus = fg._corpus()
print('finance manifold:', corpus['n_docs'], 'filings,', corpus['n_queries'], 'queries, '
      'context = top-' + str(corpus['ctx_k']))
q = fg.pick_worked_query(corpus)
print('worked query', q, '-> context facts', list(corpus['context'][q]),
      'hallucination target', int(corpus['hallu_target'][q]))

## Faithfulness is precision, groundedness is recall

An answer is a *set of atomic claims*; the context is a *set of supportable facts*. Faithfulness is the **precision** of the claims (what fraction of what was said is supported), groundedness the **recall** of the facts (what fraction of the context was used). Both are the imported set metrics — the collapse anchor is exact: a *perfect* judge's measured faithfulness equals `precision_at_k`.

In [ ]:
ans = fg.worked_answer(corpus)
print('claims:', ans['n_claims'], '| supported (oracle y):', list(ans['y']))
print(f"faithfulness (precision) = {fg.answer_faithfulness(ans):.3f}")
print(f"coverage (recall)        = {fg.answer_coverage(ans):.3f}")
print(f"F1                       = {fg.answer_f1(ans):.3f}")

## Terse versus verbose: why the two diverge

Because faithfulness counts over *claims* and coverage over *facts*, the two have different denominators and move in opposite directions. A terse answer is faithful but thin; a verbose one covers everything but invents figures. Their harmonic mean (F1) peaks at an **interior** answer length — neither a single claim nor an exhaustive one.

In [ ]:
d = fg.divergence_pair(corpus)
print(f"terse  : faithfulness {d['terse']['faithfulness']:.2f}, coverage {d['terse']['coverage']:.2f}")
print(f"verbose: faithfulness {d['verbose']['faithfulness']:.2f}, coverage {d['verbose']['coverage']:.2f}")
curve = fg.verbosity_curve(corpus)
best = max(curve, key=lambda r: r['f1'])
print('\nverbosity sweep (n_claims -> precision, coverage, F1):')
for r in curve:
    star = '  <- F1-optimal' if r is best else ''
    print(f"  n={r['n_claims']:2d}: p {r['faithfulness']:.3f}  r {r['coverage']:.3f}  f1 {r['f1']:.3f}{star}")

## Measuring with a noisy judge

On real text there is no oracle — a noisy LLM judge measures faithfulness, so the raw number is a **biased** estimate of the latent supported fraction. We audit the judge's sensitivity/specificity and invert with **Rogan–Gladen**, and we **calibrate** its confidence (ECE, Platt, isotonic) before that confidence can drive a cut.

In [ ]:
panel = fg.build_panel(corpus)
conf = fg.panel_confidence(panel)
rg = fg.panel_corrected_faithfulness(panel, conf)
print(f"naive faithfulness {rg['p_obs']:.3f}  (oracle {rg['oracle']:.3f})  -> "
      f"Rogan-Gladen corrected {rg['corrected']:.3f}")
cal = fg.panel_calibration(panel, conf)
print(f"AUC {cal['auc_raw']:.3f} | ECE raw {cal['ece_raw']:.3f} -> "
      f"Platt {cal['ece_platt']:.3f} -> isotonic {cal['ece_iso']:.3f}")

## The frontier and the back-off guarantee

Raising the confidence cut $\tau$ drops low-confidence claims — faithfulness rises, coverage falls. Trading coverage for a *guaranteed* faithfulness is the abstention frontier: **conformal risk control** (imported) fixes a threshold whose expected false-claim rate is bounded by $\alpha$. An empty retained set is an abstention — when to abstain is the next topic.

In [ ]:
cc = fg.calibrated_panel_conf(panel, conf, fg.split_panel(panel['Y'].shape[0])[0])
crc = fg.crc_backoff(panel, cc)
print(f"CRC at alpha={crc['alpha']}: cut lambda={crc['lambda_hat']:.3f}, "
      f"realized false-claim rate {crc['test_false_claim_rate']:.3f}, retaining {crc['retention']:.2f}")
fr = fg.abstention_frontier(panel, cc)
print(f"frontier: at tau=0.1 faithfulness {fr[5]['faithfulness']:.2f} / coverage {fr[5]['coverage']:.2f}; "
      f"at tau=0.8 faithfulness {fr[40]['faithfulness']:.2f} / coverage {fr[40]['coverage']:.2f}")

## Bits of grounding

A supported claim is exactly one whose **pointwise mutual information** with the context is positive: the context raises the claim's probability. A hallucination has non-positive bits — the context does not support it. This is the information-theoretic face of faithfulness (imported PMI answer model).

In [ ]:
bits = fg.panel_bits(corpus, panel)
print(f"supported claims:    mean {bits['mean_sup']:+.3f} bits")
print(f"hallucinated claims: mean {bits['mean_hal']:+.3f} bits  "
      f"({bits['frac_hal_nonpos']:.0%} have non-positive bits)")

## Verification

Every pedagogical claim is an assertion in the harness — the collapse anchors (perfect-judge faithfulness == `precision_at_k`, coverage == `recall_at_k`), the precision/recall divergence, the interior F1 optimum, the judge-overlap (non-vacuity) guard, Rogan–Gladen debiasing, the calibration drop, the monotone CRC loss, and the bits sign split.

In [ ]:
fg._run_all()
print('\nall checks passed.')